# 14 — Clean UFC Betting Odds (Kaggle → processed)

**Goal:** Turn `data/raw/kaggle_odds/UFC_betting_odds.csv` into **one row per `Fight_Id`** with decimal odds, **de-vigged** win probabilities, and documented filters.

**Input:** `../data/raw/kaggle_odds/UFC_betting_odds.csv`  
**Output:** `../data/processed/ufc_fight_odds_clean.csv`

**Rules (defaults):**
- `fight_url` → `Fight_Id` (hex slug, matches `ufc_fight_stats_cleaned.csv`).
- Keep **`region == 'us'`** (change in code if you want another market).
- Drop rows missing `odds_1` / `odds_2`.
- **De-dupe:** In this Kaggle dump, `adding_date` is often a **scrape timestamp** (not a historical “line posted at” time), so we **do not** filter on `adding_date` vs `event_date`. We keep **one row per `Fight_Id`**: the **latest `adding_date`** among `us` rows (most recent scrape / book row in the file).
- **De-vig:** `p_i_raw = 1/odds_i`, then `p_i = p_i_raw / (p1_raw + p2_raw)` so `p1 + p2 = 1`.

**Takeaway:** Notebook 15 merges this file on `Fight_Id` and compares Vegas-implied **P(favorite wins)** to the heatmap, Z-delta RF, and autoencoder+RF on the **same time-split test events**.

In [1]:
import os
import pandas as pd
import numpy as np

RAW = "../data/raw/kaggle_odds/UFC_betting_odds.csv"
OUT = "../data/processed/ufc_fight_odds_clean.csv"
FIGHTS = "../data/processed/ufc_fight_stats_cleaned.csv"

REGION_FILTER = "us"  # set to None to keep all regions

# --- Load and normalize ---------------------------------------------------
# Full Kaggle scrape: many duplicate rows per fight (books, scrape batches).
assert os.path.exists(RAW), f"Missing {RAW} — add Kaggle CSV first."

od = pd.read_csv(RAW, low_memory=False)
od["Fight_Id"] = od["fight_url"].astype(str).str.rstrip("/").str.split("/").str[-1]
od["event_date"] = pd.to_datetime(od["event_date"])
# adding_date mixes formats (microseconds vs ISO8601 "Z")
ad = pd.to_datetime(od["adding_date"], utc=True, format="mixed")
od["adding_date"] = ad.dt.tz_localize(None)

# --- Market filter ----------------------------------------------------------
if REGION_FILTER is not None:
    od = od[od["region"].astype(str).str.lower() == REGION_FILTER.lower()]

# --- Valid moneyline rows only (decimal odds > 1) ---------------------------
od = od.dropna(subset=["odds_1", "odds_2"])
od = od[(od["odds_1"] > 1) & (od["odds_2"] > 1)]

# --- Restrict to fights present in cleaned UFC stats ------------------------
# Only fights that exist in our pipeline
have = set(pd.read_csv(FIGHTS, usecols=["Fight_Id"])["Fight_Id"].astype(str))
od = od[od["Fight_Id"].isin(have)]

# One row per fight: latest adding_date in file (see markdown — scrape times, not event-relative)
# --- De-dupe to one row per Fight_Id ----------------------------------------
idx = od.groupby("Fight_Id")["adding_date"].idxmax()
clean = od.loc[idx].copy()

# --- De-vig two-way market to implied probabilities summing to 1 ------------
inv1 = 1.0 / clean["odds_1"]
inv2 = 1.0 / clean["odds_2"]
s = inv1 + inv2
clean["p_fighter_1"] = inv1 / s
clean["p_fighter_2"] = inv2 / s

keep = [
    "Fight_Id",
    "fighter_1",
    "fighter_2",
    "odds_1",
    "odds_2",
    "p_fighter_1",
    "p_fighter_2",
    "event_date",
    "adding_date",
    "source",
    "region",
]
clean = clean[keep]

# --- Persist for notebook 15 (merge on Fight_Id) ----------------------------
os.makedirs(os.path.dirname(OUT), exist_ok=True)
clean.to_csv(OUT, index=False)

print(f"Rows written: {len(clean)}  →  {OUT}")
print(clean.head(3).to_string(index=False))


Rows written: 6124  →  ../data/processed/ufc_fight_odds_clean.csv
        Fight_Id        fighter_1     fighter_2   odds_1  odds_2  p_fighter_1  p_fighter_2 event_date                adding_date     source region
0005e00b07cee542       Holly Holm  Irene Aldana 1.800000    2.03     0.530026     0.469974 2020-10-03 2025-07-27 09:10:07.798897 zewnetrzne     us
0027e179b743c86c    Jared Rosholt Josh Copeland 1.322581    3.80     0.741814     0.258186 2015-03-14 2025-07-27 09:10:07.798897 zewnetrzne     us
002921976d27b7da Alistair Overeem Stefan Struve 1.500000    2.85     0.655172     0.344828 2014-12-13 2025-07-27 09:10:07.798897 zewnetrzne     us


### QA
Spot-check merge rate vs distinct fights in cleaned stats; adjust `REGION_FILTER` or dedupe rule if coverage is too low.

In [2]:
n_fights_stats = pd.read_csv(FIGHTS, usecols=["Fight_Id"])["Fight_Id"].nunique()
print(f"Distinct Fight_Id in stats: {n_fights_stats}")
print(f"Fights with odds row after clean: {len(clean)}  ({len(clean)/n_fights_stats:.1%} of stats fights)")


Distinct Fight_Id in stats: 8482
Fights with odds row after clean: 6124  (72.2% of stats fights)
